In [2]:
import json, csv, os
from datetime import datetime


class Student:
    def __init__(self, student_id, name):
        self.student_id = student_id
        self.name = name
        self.grades = {}

    def add_grade(self, subject, score):
        self.grades[subject] = score

    def average(self):
        if not self.grades:
            return 0

        return sum(self.grades.values()) / len(self.grades)

    def letter_grade(self):
        avg = self.average()

        if avg >= 90:
            return "A"
        elif avg >= 80:
            return "B"
        elif avg >= 70:
            return "C"
        elif avg >= 60:
            return "D"
        else:
            return "F"

    def to_dict(self):
        return {
            "student_id": self.student_id,
            "name": self.name,
            "grades": self.grades
        }

    @classmethod
    def from_dict(cls, data):
        student = cls(
            data["student_id"],
            data["name"]
        )

        student.grades = data["grades"]
        return student

    def __str__(self):
        return (
            f"ID: {self.student_id} | "
            f"Name: {self.name} | "
            f"Average: {self.average():.2f} | "
            f"Grade: {self.letter_grade()}"
        )


class GradeManager:
    def __init__(self, filepath="grade_manager.json"):
        self.filepath = filepath
        self.students = {}
        self.load()

    # CRUD
    def add_student(self, name):
        student_id = len(self.students) + 1

        self.students[student_id] = Student(
            student_id,
            name
        )

        print(f"Student Added: {name}")
        self.save()

    def record_grade(self, student_id, subject, score):
        student = self.students.get(student_id)

        if student:
            student.add_grade(subject, score)
            self.save()
        else:
            print("Student not found")

    def get_student(self, student_id):
        return self.students.get(student_id)

    # Reports
    def print_report(self):
        print("\n" + "=" * 60)
        print("STUDENT REPORT")
        print("=" * 60)

        for student in self.students.values():
            print(student)

            for subject, score in student.grades.items():
                print(f"   {subject:<10}: {score}")

            print("-" * 60)

    # Persistence
    def save(self):
        data = {
            sid: student.to_dict()
            for sid, student in self.students.items()
        }

        with open(self.filepath, "w") as file:
            json.dump(data, file, indent=4)

    def load(self):
        if not os.path.exists(self.filepath):
            return

        with open(self.filepath, "r") as file:
            data = json.load(file)

        self.students = {
            int(sid): Student.from_dict(student_data)
            for sid, student_data in data.items()
        }

    # Export
    def export_csv(self, filename="grade_report.csv"):
        with open(filename, "w", newline="") as file:

            writer = csv.writer(file)

            writer.writerow([
                "Student ID",
                "Name",
                "Subject",
                "Score",
                "Average",
                "Letter Grade"
            ])

            for student in self.students.values():

                avg = round(student.average(), 2)
                grade = student.letter_grade()

                for subject, score in student.grades.items():

                    writer.writerow([
                        student.student_id,
                        student.name,
                        subject,
                        score,
                        avg,
                        grade
                    ])

        print(f"\nCSV exported successfully -> {filename}")


# Demo
manager = GradeManager("grade_manager.json")

# Add students only if file is empty
if not manager.students:

    manager.add_student("Alice")
    manager.add_student("Bob")
    manager.add_student("Charlie")
    manager.add_student("Diana")

    subjects = [
        "Maths",
        "Science",
        "English",
        "History"
    ]

    import random

    random.seed(42)

    for sid in range(1, 5):
        for subject in subjects:
            manager.record_grade(
                sid,
                subject,
                random.randint(60, 100)
            )

# Print report
manager.print_report()

# Export CSV
manager.export_csv()

# Verify CSV
print("\nCSV CONTENT\n")

with open("grade_report.csv", "r") as file:
    print(file.read())


STUDENT REPORT
ID: 1 | Name: Alice | Average: 76.25 | Grade: C
   Maths     : 100
   Science   : 67
   English   : 61
   History   : 77
------------------------------------------------------------
ID: 2 | Name: Bob | Average: 70.75 | Grade: C
   Maths     : 75
   Science   : 74
   English   : 68
   History   : 66
------------------------------------------------------------
ID: 3 | Name: Charlie | Average: 85.75 | Grade: B
   Maths     : 94
   Science   : 65
   English   : 97
   History   : 87
------------------------------------------------------------
ID: 4 | Name: Diana | Average: 65.25 | Grade: D
   Maths     : 62
   Science   : 61
   English   : 65
   History   : 73
------------------------------------------------------------

CSV exported successfully -> grade_report.csv

CSV CONTENT

Student ID,Name,Subject,Score,Average,Letter Grade
1,Alice,Maths,100,76.25,C
1,Alice,Science,67,76.25,C
1,Alice,English,61,76.25,C
1,Alice,History,77,76.25,C
2,Bob,Maths,75,70.75,C
2,Bob,Science,74,